In [ ]:
# Circular target rendering

from dataclasses import dataclass

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QBrush, QColor, QPainter, QPen


@dataclass
class CircularTaskConfig:
    """Configuration for circular target task"""

    external_radius: int = 150  # pixels
    internal_radius: int = 80  # pixels
    background_color: str = "black"
    path_color: str = "#333333"  #  darkgray < "#333333"  < "#1a1a1a" < black
    circle_border_color: str = "white"
    circle_border_width: int = 2


class CircularTargetWidget:
    """Draw circular target with tolerance band"""

    def __init__(self, config: CircularTaskConfig = None):
        self.config = config or CircularTaskConfig()

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the circular target"""
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw external circle (border)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.external_radius,
            fill_color=self.config.path_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

        # Draw internal circle (background)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.internal_radius,
            fill_color=self.config.background_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

    def _draw_filled_circle(
        self,
        painter: QPainter,
        x: int,
        y: int,
        radius: int,
        fill_color: str,
        border_color: str,
        border_width: int,
    ):
        """Helper to draw filled circle with border"""
        # Set fill color
        fill = QColor(fill_color)
        painter.setBrush(QBrush(fill))

        # Set border (pen)
        border = QColor(border_color)
        pen = QPen(border)
        pen.setWidth(border_width)
        painter.setPen(pen)

        # Draw circle
        painter.drawEllipse(x - radius, y - radius, 2 * radius, 2 * radius)

In [ ]:
# Configuration module - Window & Screen Settings

from dataclasses import dataclass
from typing import Optional

from PyQt6.QtGui import QScreen
from PyQt6.QtWidgets import QApplication, QWidget


@dataclass
class ScreenInfo:
    """Store screen information"""

    name: str
    index: int
    width: int
    height: int
    pos_x: int
    pos_y: int
    phys_width_mm: float
    phys_height_mm: float
    dpi_x: float
    dpi_y: float
    dpi_avg: float
    diag_inches: float

    def print_info(self):
        """Pretty print screen information"""
        print(f"Monitor {self.index}: {self.name}")
        print(f"  Resolution: {self.width} x {self.height} px")
        print(f"  Position: ({self.pos_x}, {self.pos_y})")
        print(
            f"  Physical Size: {self.phys_width_mm:.0f} x {self.phys_height_mm:.0f} mm "
            f'(~{self.diag_inches:.1f}")'
        )
        print(f"  DPI: {self.dpi_avg:.0f} (X: {self.dpi_x:.0f}, Y: {self.dpi_y:.0f})")


@dataclass
class WindowConfig:
    """Centralized window configuration"""

    title: str = "Wacom Pen Test - PyQt6"
    target_monitor: int = 1  # 1-indexed (1=second, 2=primary)
    width: Optional[int] = None  # None = screen width
    height: Optional[int] = None  # None = screen height
    cursor_radius: int = 16  # cursor size for circle calculations
    margin_multiplier: int = 5  # margin = margin_multiplier * cursor_radius

    def calculate_radii(self, screen_width: int, screen_height: int) -> tuple[int, int]:
        """Calculate circle radii based on screen size (matches Java logic)"""
        margin = self.margin_multiplier * self.cursor_radius
        min_dim = min(screen_width, screen_height) // 2
        external_radius = min_dim - margin
        internal_radius = external_radius - margin
        return external_radius, internal_radius


class ScreenManager:
    """Manage screen detection and configuration"""

    @staticmethod
    def get_screen_info(screen: QScreen, index: int) -> ScreenInfo:
        """Extract and compute screen information"""
        geometry = screen.geometry()
        phys_size = screen.physicalSize()

        # Calculate actual DPI from physical size and resolution
        dpi_x = (
            (geometry.width() * 25.4) / phys_size.width()
            if phys_size.width() > 0
            else 0
        )
        dpi_y = (
            (geometry.height() * 25.4) / phys_size.height()
            if phys_size.height() > 0
            else 0
        )
        dpi_avg = (dpi_x + dpi_y) / 2

        # Calculate diagonal in inches
        diag_inches = (phys_size.width() ** 2 + phys_size.height() ** 2) ** 0.5 / 25.4

        return ScreenInfo(
            name=screen.name(),
            index=index,
            width=geometry.width(),
            height=geometry.height(),
            pos_x=geometry.x(),
            pos_y=geometry.y(),
            phys_width_mm=phys_size.width(),
            phys_height_mm=phys_size.height(),
            dpi_x=dpi_x,
            dpi_y=dpi_y,
            dpi_avg=dpi_avg,
            diag_inches=diag_inches,
        )

    @staticmethod
    def get_all_screens(app: QApplication) -> list[ScreenInfo]:
        """Get info for all connected screens"""
        screens = []
        for i, screen in enumerate(app.screens(), 1):
            screens.append(ScreenManager.get_screen_info(screen, i))
        return screens

    @staticmethod
    def print_all_screens(screens: list[ScreenInfo]):
        """Print all screen information"""
        print(f"\n{'='*60}")
        print("📊 SCREEN INFORMATION")
        print(f"{'='*60}")
        print(f"Total screens detected: {len(screens)}\n")

        for screen in screens:
            screen.print_info()
            print()

    @staticmethod
    def get_target_screen(
        screens: list[ScreenInfo], config: WindowConfig
    ) -> ScreenInfo:
        """Select target screen based on config"""
        if len(screens) > 1 and config.target_monitor > 1:
            # Use target_monitor index, with fallback to primary if out of range
            idx = min(config.target_monitor - 1, len(screens) - 1)
            target = screens[idx]
        else:
            target = screens[0]

        print(f"→ Opening on Monitor {target.index}: {target.name}\n")
        return target

    @staticmethod
    def get_usable_screen_size(
        app: QApplication, screen_info: ScreenInfo
    ) -> tuple[int, int]:
        """Get usable screen dimensions accounting for taskbars/insets"""
        # Find the screen object by name
        screen_obj = None
        for screen in app.screens():
            if screen.name() == screen_info.name:
                screen_obj = screen
                break

        if screen_obj is None:
            # Fallback: use full screen dimensions
            return screen_info.width, screen_info.height

        # Get the available geometry (excludes taskbars, etc.)
        available_geo = screen_obj.availableGeometry()
        return available_geo.width(), available_geo.height()

    @staticmethod
    def get_window_drawable_area(
        window: QWidget, usable_width: int, usable_height: int
    ) -> tuple[int, int, dict]:
        """
        Calculate actual drawable area accounting for window frame insets.

        Returns:
            (actual_drawable_width, actual_drawable_height, insets_dict)
        """

        # Measure window frame insets
        frame_geo = window.frameGeometry()
        content_geo = window.geometry()

        # Calculate insets
        top_inset = content_geo.top() - frame_geo.top()
        left_inset = content_geo.left() - frame_geo.left()
        right_inset = frame_geo.right() - content_geo.right()
        bottom_inset = frame_geo.bottom() - content_geo.bottom()

        # Calculate actual drawable area
        actual_width = usable_width - left_inset - right_inset
        actual_height = usable_height - top_inset - bottom_inset

        insets = {
            "top": top_inset,
            "bottom": bottom_inset,
            "left": left_inset,
            "right": right_inset,
        }

        return actual_width, actual_height, insets

In [ ]:
# use venv wacom_test

import sys

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QPainter, QPen, QTabletEvent
from PyQt6.QtWidgets import QApplication, QWidget


class TabletTest(QWidget):
    def __init__(
        self,
        screen_info,
        circle_config,
        usable_width: int,
        usable_height: int,
        actual_drawable_width: int = None,
        actual_drawable_height: int = None,
    ):
        super().__init__()
        self.screen_info = screen_info
        self.circular_target = CircularTargetWidget(config=circle_config)
        self.usable_width = usable_width
        self.usable_height = usable_height
        # Use actual drawable dimensions if provided, otherwise use usable dimensions
        self.drawable_width = actual_drawable_width or usable_width
        self.drawable_height = actual_drawable_height or usable_height
        self.center_x = usable_width // 2
        self.center_y = usable_height // 2

    def paintEvent(self, event):
        """Draw the circular target and usable screen boundary"""
        painter = QPainter(self)
        painter.fillRect(self.rect(), Qt.GlobalColor.black)

        # Draw green rectangle showing drawable screen limits (after frame insets)
        painter.setPen(QPen(Qt.GlobalColor.green, 2))
        painter.drawRect(0, 0, self.drawable_width - 1, self.drawable_height - 1)

        # Draw the circular target
        self.circular_target.draw(painter, self.center_x, self.center_y)

    def tabletEvent(self, event: QTabletEvent):
        """Print tablet coordinates, pressure, and tilt angles"""
        print(
            f"X: {event.position().x():.1f}, Y: {event.position().y():.1f}, "
            f"Pressure: {event.pressure():.2f}, Tilt X: {event.xTilt():.1f}°, "
            f"Tilt Y: {event.yTilt():.1f}°"
        )
        event.accept()

    def mouseMoveEvent(self, event):
        """Print mouse position"""
        print(f"Mouse: X: {event.position().x():.1f}, Y: {event.position().y():.1f}")

    def mousePressEvent(self, event):
        """Print mouse click position"""
        print(
            f"Mouse click at: X={event.position().x():.1f}, Y={event.position().y():.1f}"
        )


# ===== CONFIGURATION =====
config = WindowConfig(
    title="Wacom Pen Test - PyQt6",
    target_monitor=2,  # 1=primary, 2=second monitor, etc.
    cursor_radius=16,
    margin_multiplier=5,
)

# ===== SETUP =====
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

# Get screen information
screens = ScreenManager.get_all_screens(app)
ScreenManager.print_all_screens(screens)

# Get target screen
target_screen_info = ScreenManager.get_target_screen(screens, config)

# Get usable screen size (accounts for taskbars and insets)
usable_width, usable_height = ScreenManager.get_usable_screen_size(
    app, target_screen_info
)

# Step 1: Calculate initial circle radii based on usable screen size (Java logic)
external_radius, internal_radius = config.calculate_radii(
    screen_width=usable_width,
    screen_height=usable_height,
)
print(f"Cursor radius: {config.cursor_radius} px")
print(
    f"Circle margin: {config.margin_multiplier} × {config.cursor_radius} = "
    f"{config.margin_multiplier * config.cursor_radius} px"
)
print(f"External radius: {external_radius} px")
print(f"Internal radius: {internal_radius} px\n")

# Step 2: Create circle config with initial radii
circle_config = CircularTaskConfig(
    external_radius=external_radius,
    internal_radius=internal_radius,
)

# Step 3: Create and configure window
w = TabletTest(target_screen_info, circle_config, usable_width, usable_height)
w.setWindowTitle(config.title)

# Position and size window
w.move(target_screen_info.pos_x, target_screen_info.pos_y)
window_width = config.width if config.width else usable_width
window_height = config.height if config.height else usable_height
w.resize(window_width, window_height)

# Step 4: Show window and let Qt calculate frame insets
w.show()
app.processEvents()

# Step 5: Measure actual drawable area accounting for window frame insets
actual_width, actual_height, insets = ScreenManager.get_window_drawable_area(
    w, usable_width, usable_height
)
print(
    f"\nWindow frame insets: Top={insets['top']}, Bottom={insets['bottom']}, Left={insets['left']}, Right={insets['right']}"
)

# Step 6: Recalculate radii based on actual drawable area
external_radius, internal_radius = config.calculate_radii(actual_width, actual_height)

# Step 7: Create new config with corrected radii
corrected_circle_config = CircularTaskConfig(
    external_radius=external_radius,
    internal_radius=internal_radius,
)

# Step 8: Update widget with corrected drawable dimensions and circle config
w.circular_target = CircularTargetWidget(config=corrected_circle_config)
w.drawable_width = actual_width
w.drawable_height = actual_height
w.center_x = actual_width // 2
w.center_y = actual_height // 2
w.update()


# Step 9: Finalize window display
w.raise_()
w.activateWindow()
w.setFocus()

print(f"{'='*60}\n")
app.exec()